## 1. Question

Does moving the level gate from 30 to 40 affect player retention? Players are randomly assigned to Gate 30 (control) or Gate 40 (treatment), and the analysis compares Day-1 and Day-7 retention between the two groups.

In [13]:
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path.cwd().parent / "src"))

from plots import retention_chart, diff_chart

## 2. Data

The Cookie Cats A/B test contains roughly 90K players, assigned to Gate 30 or Gate 40. Basic data-quality checks are run on the loaded data before analysing retention, so that the retention rates that drive every test below rest on a table of the expected size and shape.

In [14]:
with open("../sql/02_retention.sql", encoding="utf-8") as f:
    query = f.read()

result = pd.read_sql_query(query, con)
result

,version,n,ret1_count,ret7_count,ret1_rate,ret7_rate
0,gate_30,44700,20034,8502,0.448188,0.190201
1,gate_40,45489,20119,8279,0.442283,0.182000


## 3. Retention Tests

Day-1 and Day-7 retention are compared between Gate 30 and Gate 40 with two-proportion z-tests, with the effect reported as an absolute (percentage-point) and relative difference. Uncertainty is quantified with 95% confidence intervals: bootstrap intervals for the Gate 40 − Gate 30 difference, resampled from the same observed retention counts. A power calculation shows the sample size needed to detect a 1 percentage-point change.

In [15]:
from statsmodels.stats.proportion import proportions_ztest

g30 = result[result.version == "gate_30"].iloc[0]
g40 = result[result.version == "gate_40"].iloc[0]

for metric in ["ret1_count", "ret7_count"]:
    stat, p = proportions_ztest(
        [g30[metric], g40[metric]],
        [g30["n"], g40["n"]]
    )
    print(metric, "z =", round(stat, 3), "p =", round(p, 4))

ret1_count z = 1.784 p = 0.0744
ret7_count z = 3.164 p = 0.0016


In [16]:
import numpy as np

rng = np.random.default_rng(42)

def boot_diff(c1, n1, c2, n2, B=10000):
    p1 = rng.binomial(n1, c1/n1, B) / n1
    p2 = rng.binomial(n2, c2/n2, B) / n2
    d = p2 - p1
    return np.percentile(d, [2.5, 97.5])

lo, hi = boot_diff(
    g30["ret7_count"], g30["n"],
    g40["ret7_count"], g40["n"]
)

print("95% CI for diff (gate_40 - gate_30):", lo, hi)

95% CI for diff (gate_40 - gate_30): -0.01335625406009358 -0.0031113273051778362


In [17]:
diff_pp = (g40["ret7_rate"] - g30["ret7_rate"]) * 100
rel = (g40["ret7_rate"] / g30["ret7_rate"] - 1) * 100

print(f"Absolute difference: {diff_pp:.2f} percentage points")
print(f"Relative change: {rel:.2f}%")

Absolute difference: -0.82 percentage points
Relative change: -4.31%


In [18]:
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize

es = proportion_effectsize(
    g30["ret7_rate"] + 0.01,
    g30["ret7_rate"]
)

n_needed = NormalIndPower().solve_power(
    es,
    alpha=0.05,
    power=0.8
)

print("Per-group n for 80% power to detect +1pp:", int(n_needed))

Per-group n for 80% power to detect +1pp: 24657


## 4. Robustness

The retention comparison is repeated after removing the top 1% of players by `sum_gamerounds`. These are a small number of very heavy players, and the check shows whether the conclusion depends on such extreme observations or holds for the bulk of the sample.

In [19]:
with open("../sql/03_retention_outliers.sql", encoding="utf-8") as f:
    trimmed = pd.read_sql_query(f.read(), con)

t30 = trimmed[trimmed.version == "gate_30"].iloc[0]
t40 = trimmed[trimmed.version == "gate_40"].iloc[0]

print(
    f"Day-7 difference after dropping top 1%: "
    f"{(t40['ret7_rate'] - t30['ret7_rate']) * 100:.2f} pp"
)

Day-7 difference after dropping top 1%: -0.83 pp


In [20]:
lo1, hi1 = boot_diff(
    g30["ret1_count"], g30["n"],
    g40["ret1_count"], g40["n"]
)

print("95% CI for Day-1 diff (gate_40 - gate_30):", lo1, hi1)

95% CI for Day-1 diff (gate_40 - gate_30): -0.0125001351089968 0.000652748927722181


## 5. Sample Ratio Mismatch (SRM)

The observed Gate 30 / Gate 40 allocation is checked against the 50/50 split the experiment intended, using a chi-square goodness-of-fit test. The result is a mild imbalance of unknown cause: the two groups differ slightly in size, which is worth monitoring, but it is not on its own evidence that the experiment pipeline is broken.

In [21]:
from scipy.stats import chisquare

observed = [g30["n"], g40["n"]]

stat, p = chisquare(observed)

print("Chi-square =", round(stat, 3))
print("p-value =", round(p, 4))

Chi-square = 6.902
p-value = 0.0086


## 6. Charts

The figures summarise retention by group and the Gate 40 − Gate 30 difference. The group chart shows Day-1 and Day-7 retention with 95% Wilson confidence intervals; the difference chart shows the Gate 40 − Gate 30 gap in percentage points with bootstrap confidence intervals, against a zero reference line.

In [22]:
import pandas as pd

retention_df = pd.read_sql("""
SELECT
    version,
    COUNT(*) AS n,
    SUM(CASE WHEN retention_1 = 1 THEN 1 ELSE 0 END) AS ret1_count,
    SUM(CASE WHEN retention_7 = 1 THEN 1 ELSE 0 END) AS ret7_count
FROM players
GROUP BY version
ORDER BY version;
""", con)

retention_df

,version,n,ret1_count,ret7_count
0,gate_30,44700,20034,8502
1,gate_40,45489,20119,8279


In [23]:
import sys
sys.path.append("../src")
from plots import retention_chart, diff_chart

retention_chart(
    retention_df,
    outdir="../reports/figures"
)

d1 = (g40["ret1_rate"] - g30["ret1_rate"]) * 100
d7 = (g40["ret7_rate"] - g30["ret7_rate"]) * 100

diff_chart(
    {
        "Day-1": (d1, lo1 * 100, hi1 * 100),
        "Day-7": (d7, lo * 100, hi * 100),
    },
    outdir="../reports/figures"
)

print("Charts generated successfully.")

Charts generated successfully.


In [24]:
from pathlib import Path

figures = Path("../reports/figures")

print(list(figures.glob("*.png")))

[WindowsPath('../reports/figures/retention_by_group.png'), WindowsPath('../reports/figures/retention_difference.png')]


## 7. Conclusion

Gate 40 reduces Day-7 retention relative to Gate 30: the drop is about 0.8 percentage points (roughly 4% relative), and the 95% bootstrap interval for the difference lies entirely below zero. The Day-1 difference is smaller and not statistically significant.

The Day-7 effect persists after removing the top 1% of players by game rounds, so it is not an artefact of extreme observations. Gate 30 is therefore the recommended configuration.